**Note: When splitting a county into two model areas use following convention:**
- CO_FIPS = 9057 (9=added table + 0=SubareaID + 57=CO_FIPS)
- CO_FIPS = 9157 (9=added table + 1=SubareaID + 57=CO_FIPS)

In [75]:
import pandas as pd
import re
import numpy as np

In [76]:
category_dict = {
  'POP': ['TOTPOP','GQPOP','HHPOP','POP0017','POP1864','POP65P'],
  'HH': ['HH'],
  'RETL': ['RETL','FOOD'],
  'IND': ['MANU','WSLE'],
  'OFFI': ['OFFI','GVED','HLTH','OTHR']
}

cat_expanded_df = (
    pd.DataFrame([
        {'CATEGORY': cat, 'VARIABLE': v}
        for cat, vars in category_dict.items()
        for v in vars
    ])
)

cat_expanded_df

,CATEGORY,VARIABLE
0,POP,TOTPOP
1,POP,GQPOP
2,POP,HHPOP
3,POP,POP0017
4,POP,POP1864
5,POP,POP65P
6,HH,HH
7,RETL,RETL
8,RETL,FOOD
9,IND,MANU


In [77]:
subarea_df = pd.DataFrame([
  ['WF'  ,1], # WF TDM
  ['UDOT',0]  # USTM
], columns=("MODEL","SUBAREAID"))
subarea_df

,MODEL,SUBAREAID
0,WF,1
1,UDOT,0


In [78]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


# Split Box Elder and Weber counties

In [79]:
df_box = pd.read_csv(
    "data/From-Josh-20251121/REMM-WF-Model-Area-Splits (RTP27) - Box Elder.csv"
).drop(columns=['TOTALEMP_PCT_UDOT'])
df_box["CO_NAME"] = "BOX ELDER"

df_weber = pd.read_csv(
    "data/From-Josh-20251121/REMM-WF-Model-Area-Splits (RTP27) - Weber.csv"
).drop(columns=['TOTAL_PCT_WF','TOTAL_PCT_UDOT'])
df_weber["CO_NAME"] = "WEBER"

wf_model_area_splits_df = pd.concat([df_box, df_weber], ignore_index=True)

wf_model_area_splits_df.rename(columns={'Year':'YEAR'}, inplace=True)

# add underscore before PCT where missing
wf_model_area_splits_df.columns = [
    re.sub(r'(?<!_)PCT', r'_PCT', col)
    for col in wf_model_area_splits_df.columns
]

wf_model_area_splits_df

,YEAR,HH_PCT_WF,HH_PCT_UDOT,POP_PCT_WF,POP_PCT_UDOT,OFFI_PCT_WF,OFFI_PCT_UDOT,RETL_PCT_WF,RETL_PCT_UDOT,IND_PCT_WF,IND_PCT_UDOT,CO_NAME
0,2023,0.541,0.459,0.530,0.470,0.714,0.286,0.661,0.339,0.378,0.622,BOX ELDER
1,2024,0.541,0.459,0.529,0.471,0.713,0.287,0.665,0.335,0.381,0.619,BOX ELDER
2,2025,0.540,0.460,0.528,0.472,0.713,0.287,0.668,0.332,0.383,0.617,BOX ELDER
3,2026,0.539,0.461,0.528,0.472,0.713,0.287,0.671,0.329,0.386,0.614,BOX ELDER
4,2027,0.539,0.461,0.527,0.473,0.713,0.287,0.673,0.327,0.388,0.612,BOX ELDER
...,...,...,...,...,...,...,...,...,...,...,...,...
81,2061,0.961,0.039,0.967,0.033,0.962,0.038,1.001,-0.001,0.994,0.006,WEBER
82,2062,0.961,0.039,0.967,0.033,0.962,0.038,1.001,-0.001,0.994,0.006,WEBER
83,2063,0.961,0.039,0.967,0.033,0.962,0.038,1.001,-0.001,0.994,0.006,WEBER
84,2064,0.961,0.039,0.967,0.033,0.962,0.038,1.002,-0.002,0.994,0.006,WEBER


In [80]:
wf_model_area_splits_long_df = wf_model_area_splits_df.melt(
    id_vars=["CO_NAME","YEAR"],
    var_name="COLNAME",
    value_name="PERCENT"
)

parts = wf_model_area_splits_long_df["COLNAME"].str.split("_", expand=True)

wf_model_area_splits_long_df["CATEGORY"] = parts[0]
wf_model_area_splits_long_df["MODEL"] = parts[2]

wf_model_area_splits_long_df.drop(columns=['COLNAME'], inplace=True)

wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(
    cat_expanded_df,
    on="CATEGORY",
    how="left"
)

# add subarea id
wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(subarea_df,on='MODEL')

# add CO_FIPS
wf_model_area_splits_long_df = wf_model_area_splits_long_df.merge(counties_df,on='CO_NAME')

# CO_FIPS_new 
wf_model_area_splits_long_df['CO_FIPS_new'] = 9000 + wf_model_area_splits_long_df['SUBAREAID'] * 100 + wf_model_area_splits_long_df['CO_FIPS']

wf_model_area_splits_long_df

,CO_NAME,YEAR,PERCENT,CATEGORY,MODEL,VARIABLE,SUBAREAID,CO_FIPS,CO_FIPS_new
0,BOX ELDER,2023,0.541,HH,WF,HH,1,3,9103
1,BOX ELDER,2024,0.541,HH,WF,HH,1,3,9103
2,BOX ELDER,2025,0.540,HH,WF,HH,1,3,9103
3,BOX ELDER,2026,0.539,HH,WF,HH,1,3,9103
4,BOX ELDER,2027,0.539,HH,WF,HH,1,3,9103
...,...,...,...,...,...,...,...,...,...
2575,WEBER,2063,0.006,IND,UDOT,WSLE,0,57,9057
2576,WEBER,2064,0.006,IND,UDOT,MANU,0,57,9057
2577,WEBER,2064,0.006,IND,UDOT,WSLE,0,57,9057
2578,WEBER,2065,0.006,IND,UDOT,MANU,0,57,9057


# Population

In [81]:
pop_df = pd.concat([pd.read_csv('intermediate/tdm_population_by_county_2023.csv'),pd.read_csv('intermediate/tdm_population_by_county_2025+.csv')]).rename(columns={'TDM_VARIABLE':'VARIABLE'})
pop_df

,CO_FIPS,YEAR,VARIABLE,VALUE,CO_NAME
0,1,2023,TOTPOP,7314,BEAVER
1,3,2023,TOTPOP,61251,BOX ELDER
2,5,2023,TOTPOP,141700,CACHE
3,7,2023,TOTPOP,20654,CARBON
4,9,2023,TOTPOP,998,DAGGETT
...,...,...,...,...,...
8318,57,2061,POP65P,88319,WEBER
8319,57,2062,POP65P,90098,WEBER
8320,57,2063,POP65P,91765,WEBER
8321,57,2064,POP65P,93217,WEBER


In [82]:
# 1. Join on CO_FIPS, VARIABLE, YEAR
merged = pop_df.merge(
    wf_model_area_splits_long_df,
    on=["CO_FIPS", "VARIABLE", "YEAR"],
    how="left",  # keep all original rows
    suffixes=("", "_split")
)

# 2. Choose which CO_FIPS to use
#    If there is a split row, use CO_FIPS_new; otherwise use original CO_FIPS
merged["CO_FIPS_final"] = merged["CO_FIPS_new"].fillna(merged["CO_FIPS"])

# 3. Choose which VALUE to use
#    If there is a split (PERCENT not null), multiply VALUE by PERCENT
#    (Assumes PERCENT is a 0–1 fraction; if it's 0–100, divide by 100)
merged["VALUE_final"] = np.where(
    merged["PERCENT"].notna(),
    merged["VALUE"] * merged["PERCENT"],   # or VALUE * PERCENT / 100
    merged["VALUE"]
)

# do banker's roundign to calculate HBJ counts
merged['VALUE_final'] = np.round(merged['VALUE_final']).astype(int)

# 4. Keep only what you need
pop_split_df = merged[["CO_FIPS_final", "VARIABLE", "YEAR", "VALUE_final"]].rename(
    columns={"CO_FIPS_final": "CO_FIPS", "VALUE_final": "VALUE"}
)

pop_split_df['CO_FIPS'] = pop_split_df['CO_FIPS'].astype(int)

pop_split_df = pop_split_df.sort_values(['CO_FIPS','YEAR'])
pop_split_df = pop_split_df[['CO_FIPS','YEAR','VARIABLE','VALUE']]
pop_split_df

,CO_FIPS,YEAR,VARIABLE,VALUE
0,1,2023,TOTPOP,7314
31,1,2023,GQPOP,299
62,1,2023,HHPOP,7015
93,1,2023,POP0017,1942
124,1,2023,POP1864,4239
...,...,...,...,...
4028,9157,2065,HHPOP,367411
5299,9157,2065,HH,164591
6570,9157,2065,POP0017,59084
7841,9157,2065,POP1864,222289


# Employment

In [83]:
emp_df = pd.read_csv('intermediate/tdm_employment_by_county_with_hbj.csv').rename(columns={'TDM_SECTOR':'VARIABLE'})
emp_df

,CO_FIPS,YEAR,VARIABLE,EMPCOUNT,HBJPCT,HBJ,NONHBJ,CO_NAME
0,1,2023,AGRI,738,0.011,8,730,BEAVER
1,1,2023,CONS,278,0.051,14,264,BEAVER
2,1,2023,FOOD,517,0.012,6,511,BEAVER
3,1,2023,GVED,810,0.074,60,750,BEAVER
4,1,2023,HLTH,160,0.074,12,148,BEAVER
...,...,...,...,...,...,...,...,...
314,57,2023,MING,210,0.026,5,205,WEBER
315,57,2023,OFFI,11496,0.054,621,10875,WEBER
316,57,2023,OTHR,39215,0.027,1059,38156,WEBER
317,57,2023,RETL,16106,0.023,370,15736,WEBER


In [84]:
# 1. Join on CO_FIPS, VARIABLE, YEAR
merged = emp_df.merge(
    wf_model_area_splits_long_df,
    on=["CO_FIPS", "VARIABLE", "YEAR"],
    how="left",  # keep all original rows
    suffixes=("", "_split")
)

# 2. Choose which CO_FIPS to use
#    If there is a split row, use CO_FIPS_new; otherwise use original CO_FIPS
merged["CO_FIPS_final"] = merged["CO_FIPS_new"].fillna(merged["CO_FIPS"])

# 3. Choose which VALUE to use
#    If there is a split (PERCENT not null), multiply VALUE by PERCENT
#    (Assumes PERCENT is a 0–1 fraction; if it's 0–100, divide by 100)
merged["NONHBJ_final"] = np.where(
    merged["PERCENT"].notna(),
    merged["NONHBJ"] * merged["PERCENT"],   # or NONHBJ * PERCENT / 100
    merged["NONHBJ"]
)
merged["HBJ_final"] = np.where(
    merged["PERCENT"].notna(),
    merged["HBJ"] * merged["PERCENT"],   # or HBJ * PERCENT / 100
    merged["HBJ"]
)

# do banker's roundign to calculate HBJ counts
merged['NONHBJ_final'] = np.round(merged['NONHBJ_final']).astype(int)
merged['HBJ_final'] = np.round(merged['HBJ_final']).astype(int)

# 4. Keep only what you need
emp_split_df = merged[["CO_FIPS_final", "VARIABLE", "YEAR", "NONHBJ_final", "HBJ_final"]].rename(
    columns={"CO_FIPS_final": "CO_FIPS", "NONHBJ_final": "NONHBJ", "HBJ_final": "HBJ"}
)

emp_split_df['CO_FIPS'] = emp_split_df['CO_FIPS'].astype(int)

emp_split_df

,CO_FIPS,VARIABLE,YEAR,NONHBJ,HBJ
0,1,AGRI,2023,730,8
1,1,CONS,2023,264,14
2,1,FOOD,2023,511,6
3,1,GVED,2023,750,60
4,1,HLTH,2023,148,12
...,...,...,...,...,...
330,9057,OTHR,2023,992,28
331,9157,RETL,2023,15516,365
332,9057,RETL,2023,220,5
333,9157,WSLE,2023,10204,229


In [85]:
# NONHBJ
emp_split_nonhbj_df = emp_split_df[['CO_FIPS','YEAR','VARIABLE','NONHBJ']].copy()
emp_split_nonhbj_df.rename(columns={'NONHBJ':'VALUE'}, inplace=True)
emp_split_nonhbj_df

,CO_FIPS,YEAR,VARIABLE,VALUE
0,1,2023,AGRI,730
1,1,2023,CONS,264
2,1,2023,FOOD,511
3,1,2023,GVED,750
4,1,2023,HLTH,148
...,...,...,...,...
330,9057,2023,OTHR,992
331,9157,2023,RETL,15516
332,9057,2023,RETL,220
333,9157,2023,WSLE,10204


In [86]:
# HBJ
emp_split_hbj_df = emp_split_df.groupby(['CO_FIPS','YEAR'], as_index=False).agg(VALUE=('HBJ','sum'))
emp_split_hbj_df['VARIABLE'] = 'HBJ'
emp_split_hbj_df = emp_split_hbj_df[['CO_FIPS','YEAR','VARIABLE','VALUE']]
emp_split_hbj_df

,CO_FIPS,YEAR,VARIABLE,VALUE
0,1,2023,HBJ,244
1,3,2023,HBJ,190
2,5,2023,HBJ,2518
3,7,2023,HBJ,155
4,9,2023,HBJ,13
5,11,2023,HBJ,5888
6,13,2023,HBJ,451
7,15,2023,HBJ,99
8,17,2023,HBJ,121
9,19,2023,HBJ,535


In [87]:
emp_split_combined_df = pd.concat([emp_split_nonhbj_df, emp_split_hbj_df])
emp_split_combined_df = emp_split_combined_df.sort_values(['CO_FIPS','YEAR','VARIABLE'])
emp_split_combined_df

,CO_FIPS,YEAR,VARIABLE,VALUE
0,1,2023,AGRI,730
1,1,2023,CONS,264
2,1,2023,FOOD,511
3,1,2023,GVED,750
0,1,2023,HBJ,244
...,...,...,...,...
324,9157,2023,MANU,19126
327,9157,2023,OFFI,10592
329,9157,2023,OTHR,37164
331,9157,2023,RETL,15516


In [88]:
pop_split_df.to_csv('intermediate/split_pop.csv', index=False)
emp_split_combined_df.to_csv('intermediate/split_emp.csv', index=False)